# SALM Fine-tuning 02 · Stop answering in Spanish: English → Catalan speech translation

> **Task** English speech → Catalan text · **Data** CoVoST 2 `en_ca` (114 h used) · **Metric** BLEU ↑
> **Result** 15.21 → **27.84** BLEU (+83.0%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: close, but in the wrong language

The base model already translates, even though it was trained for transcription. Into Catalan, though, it struggles:

```
EN    She recovered again briefly, but then suffered another relapse.
REF   Es va tornar a recuperar breument, però després va patir una altra recaiguda.
BASE  Recuperàs de nou a peu, pero després va sufrir qualsevol altra recaída.
      ← Spanish: "pero", "sufrir", "recaída"
```

It knows Catalan, but Spanish keeps leaking in. The goal of this tutorial is a translator that stays in Catalan,
without hurting English transcription.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data

CoVoST 2's English audio is identical across every `en_X` direction, so the script decodes it once and reads only the
`translation` column. Two of the eight train shards give **72,358 utterances (114 h)**. Dev is 1,500 random validation
utterances; test is the full official set of 15,531.

In [ ]:
d = M / "covost2"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py covost2 --tgt ca --train-shards 2", log="prep_covost2.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import random
V = [json.loads(l) for l in open(d / "en_ca_validation.json")]
random.Random(0).shuffle(V)
(d / "en_ca_dev1500.json").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in V[:1500]))
for s in ("train", "dev1500", "test"):
    rows = [json.loads(l) for l in open(d / f"en_ca_{s}.json")]
    print(f"{s:8s} {len(rows):6d} utterances, {sum(r['duration'] for r in rows) / 3600:6.1f} h")

## 2. Baseline: don't say "not Spanish"

Three prompts, each scored on dev and test (sacreBLEU with 13a tokenization; chrF alongside).

In [ ]:
TASK = "--task bleu --bleu-tokenize 13a"
PROMPTS = {
    "p1": "Translate the English speech into Catalan.",
    "p2": "Translate the English audio into Catalan (català). Output only the Catalan translation.",
    "p3": "Listen to the English speech and translate it into Catalan, not Spanish. Output only the translation.",
}
base = {}
for p, text in PROMPTS.items():
    for s, man in (("dev", "en_ca_dev1500"), ("test", "en_ca_test")):
        r = evaluate(CKPT, d / f"{man}.json", f"st_base_{p}_{s}", TASK, f"--prompt {q(text)}")
        base[p, s] = r["score"]
    print(f"{p}  dev BLEU {base[p, 'dev']:5.2f}   test BLEU {base[p, 'test']:5.2f}   {text}")
BEST_P = max(PROMPTS, key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"] for p in PROMPTS)  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f} BLEU")

Telling the model *not* to use Spanish makes it **worse**: mentioning the unwanted language primes it. p2 is the
prompt chosen on dev (18.33 BLEU). The baseline is its test score, **15.21** BLEU, which is also the test-oracle. p2 is
the training prompt.

How often does Spanish leak in? A rough measure: the share of outputs that contain a Spanish-only marker.

In [ ]:
import re
SPANISH = re.compile(r"ñ|¿|¡|\b(los|las|muy|pero|ella|está|también|qué|cómo|hay|es un|una vez)\b", re.I)
def spanish_rate(name, key="pred_text"):
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    return sum(bool(SPANISH.search(r[key])) for r in rows) / len(rows)
print(f"references {spanish_rate('st_base_p2_dev', 'text'):.2%}   base model {spanish_rate('st_base_p2_dev'):.2%}")
pairs = " ".join(f"{d}/en_ca_{s}.json {d}/en_ca_{s}_ft.json" for s in ("train", "dev1500", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(PROMPTS['p2'])} {pairs}")

## 3. Config: leave the ears alone

The audio is English, which the model hears perfectly. The problem is entirely on the *output* side, so this is the one
recipe in the series that **doesn't train the encoder**: only the projection and the LoRA adapters learn (no
`--train-encoder`). 114 h is enough data to batch for throughput: 64k tokens per batch, ~147 batches per epoch,
400 steps ≈ 2.7 epochs.

In [ ]:
CFG = WORK / "configs" / "catalan.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'catalan'} "
   f"--train-manifest {d}/en_ca_train_ft.json --val-manifest {d}/en_ca_dev1500_ft.json --val-name ca_dev "
   f"--train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 64000 --limit-train-batches 100 --limit-val-batches 25 --save-top-k -1")

## 4. Train

About 5 s per step, 35 minutes. Expect `val_loss` to fall from ~1.38 (step 100) to ~1.25 (step 300).

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 200 300 400")

## 5. Select on dev

In [ ]:
dev = {}
for step in (200, 300, 400):
    out = EXPORT / f"catalan_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'catalan'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_catalan_{step}.log")
    r = evaluate(out, d / "en_ca_dev1500_ft.json", f"catalan_s{step}_dev", TASK)
    dev[step] = r["score"]
    print(f"step {step}: dev BLEU {r['score']:.2f}  chrF {r['chrf']:.2f}")
BEST = max(dev, key=dev.get)
print("selected step", BEST)

Expected dev BLEU: step 200 → 32.23, **step 300 → 32.80**, step 400 → 32.24 (base: 18.33). The selection rule, the
best dev BLEU, is fixed before looking at the numbers.

## 6. Test, once, plus the English guard

In [ ]:
MODEL = EXPORT / f"catalan_s{BEST}"
ft = evaluate(MODEL, d / "en_ca_test_ft.json", "catalan_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "catalan_librispeech", "--task asr --normalizer english")
print(f"test BLEU: base {BASELINE:.2f} → fine-tuned {ft['score']:.2f} ({(ft['score'] - BASELINE) / BASELINE:+.1%}); chrF {ft['chrf']:.2f}")
print(f"Spanish-marker rate: {spanish_rate('catalan_TEST'):.2%}")
print(f"LibriSpeech WER: {100 * guard['score']:.2f}%")
samples("catalan_TEST", n=3, keys=("source", "text", "pred_text"))
result("02 baseline test BLEU", BASELINE, 15.21, 0.5)
result("02 fine-tuned test BLEU", ft["score"], 27.84, 1.0)

Expected results:

| | test BLEU | test chrF | LibriSpeech WER |
|---|---:|---:|---:|
| base, prompt chosen on dev (p2) | 15.21 | 43.65 | 1.54% |
| **fine-tuned** | **27.84** | **53.90** | **1.45%** |

**+83.0% relative.** The Spanish-marker rate drops from 8.9% (base model, dev) to 1.8% (fine-tuned, test), close to
the human references (0.87%). With the encoder frozen, English ASR is untouched.

```
EN    My first day at work was horrible, but it was not my fault
BASE  El meu primer dia de treball fue horrible, pero no fue mi culpa.     ← drifts into Spanish
FT    El meu primer dia de treball va ser horrible, però no va ser la meva culpa.
```

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="catalan")

## Takeaways

1. **Check what the model already does.** It translates before any fine-tuning. Pick a direction with a *specific*
   failure you can measure.
2. **Freeze the encoder when the shift is in the output.** Faster, and English survives unharmed.
3. **Don't name the thing you want to avoid** in a prompt.